<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T02-L02-pytorch-core-idioms/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T02-L02-pytorch-core-idioms/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T02-L02-pytorch-core-idioms/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have. The cell below installs `torch==2.14.1`, and does nothing where
they are already present. On Kaggle, switch Internet on in the notebook's settings first;
Kaggle allows that only for phone-verified accounts.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = [("torch", "torch==2.14.1")]            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T02-L02-pytorch-core-idioms/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


# PyPI's Linux wheel of torch brings gigabytes of CUDA libraries with it. On Linux this cell
# installs the CPU build from PyTorch's own package index instead, the build this lesson was
# checked with; on macOS and Windows the wheel on PyPI is a CPU build already.
COMMONS_INDEX = {"torch": "https://download.pytorch.org/whl/cpu"}
if not sys.platform.startswith("linux"):
    COMMONS_INDEX = {}

_missing = [(imp, pip) for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(pip for imp, pip in _missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        _install = [sys.executable, "-m", "pip", "install", "-q"]
    else:
        _install = ["uv", "pip", "install", "-q", "--python", sys.executable]
    # one install per package index ("" is PyPI)
    for _index in sorted({COMMONS_INDEX.get(imp, "") for imp, pip in _missing}):
        subprocess.run([*_install, *(["--index-url", _index] if _index else []),
                        *[pip for imp, pip in _missing if COMMONS_INDEX.get(imp, "") == _index]],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T02-L02 · PyTorch core idioms: a training loop whose checkpoint survives a restart

**You will build:** a small PyTorch model, a seeded data loader and a training loop; then you
"kill" the loop part-way through, bring it back from a checkpoint in brand-new objects, and
measure, with exact tensor equality, what a resume that forgets one piece of state costs.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, no GPU; the only download is PyTorch itself, if you do
not already have it · **Prerequisites:** T02-L01-autograd-from-scratch

T02-L01 built the backward pass by hand. This lesson uses PyTorch's own, and the idioms every
later PyTorch notebook and repository takes for granted: tensors and views, `nn.Module`, a
`DataLoader`, an optimiser and a scheduler, `state_dict()`. It ends on a place where real
training code goes quietly wrong: restarting a job that died.

By the end you will be able to:

1. Implement a test for whether two tensors share memory, and measure which tensor operations
   return views and which return copies.
2. Implement a device probe that checks at run time what is available, and explain why the
   answer PyTorch gives at build time is not enough.
3. Implement an `nn.Module` with parameters, a persistent buffer, a non-persistent buffer and
   dropout, and measure what `state_dict()` holds and what it leaves out.
4. Implement a seeded `DataLoader` and a training loop (gradient clearing, forward pass, loss,
   backward pass, gradient clipping, optimiser step, scheduler step) whose order is checked by
   probes rather than by comparing numbers.
5. Implement checkpoint capture and restore, kill a run at chosen steps, and measure with exact
   tensor equality which missing piece of state makes the resumed run differ from the run that
   was never interrupted.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T02-L01`

**This lesson assumes you can already:**

- write Python classes with `__init__` and methods, and pass a function as an argument
- use numpy arrays: read a shape, take a slice, and know when an operation returns a view instead of a copy
- describe a training loop in order: forward pass, loss, backward pass, parameter update (T02-L01 builds the backward pass)
- run a notebook on Linux, Windows or an Apple-silicon Mac with macOS 14 or later: the pinned PyTorch wheel has no build for Intel Macs or older macOS

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) · PyTorch tutorials · *free* — tensors, data and a training loop on a small model: the picture for exercises 1, 3 and 5
- [Saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html) · PyTorch tutorials · *free* — state_dict basics, and what it leaves out that exercise 6 measures
- [Reproducibility](https://docs.pytorch.org/docs/2.14/notes/randomness.html) · PyTorch documentation, notes · *free* — what the docs promise and do not, for exercises 4 and 6
- [Serialization semantics](https://docs.pytorch.org/docs/2.14/notes/serialization.html) · PyTorch documentation, notes · *free* — what weights_only=True accepts, for exercise 6
- [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · NumPy user guide · *free* — the shape rule behind the (B, 1) against (B,) trap in exercise 5

In [ ]:
# Setup, part 1. PyTorch is imported here, at the top level and unguarded, so the launcher cell
# above knows this lesson needs it and installs the pinned version when it is missing.
import sys

import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__.split("+")[0])
print(f"platform {sys.platform}: torch build {torch.__version__}")

# Feature detection, not version parsing: the PyTorch you run may be one that was already installed, and
# what matters is whether the one function exercise 2 builds on exists in it.
import inspect

_HAS_ACCELERATOR_API = (
    hasattr(torch, "accelerator")
    and "check_available" in inspect.signature(torch.accelerator.current_accelerator).parameters)
if not _HAS_ACCELERATOR_API:
    print("this PyTorch has no torch.accelerator.current_accelerator(check_available=...), so exercise 2 "
          "is skipped; every other exercise runs.")

In [ ]:
# Setup, part 2: the rest of the imports, the fixed protocol, and the notebook's bookkeeping.
import contextlib
import copy
import functools
import gc
import io
import math
import pickle
import random
import warnings
import weakref
from types import SimpleNamespace
from typing import Callable

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, RandomSampler

# One thread: the equality this lesson measures was measured under this setting, on this machine
# and this PyTorch build. It is a measurement, not a promise (see the sources at the end).
torch.set_num_threads(1)

# The fixed protocol. Every number the exercises and the prose refer to lives here, and the cell
# after this one prints it.
SEED = 7                       # the one seed of the experiment
R = 224                        # rows in the table
B = 32                         # rows in a batch
E = math.ceil(R / B)           # batches in an epoch (drop_last=False keeps a short last batch, if there is one)
N = 40                         # training steps in a whole run
C = 4                          # a checkpoint is taken every C steps
L = math.lcm(C, E)             # the first step at which a checkpoint lands exactly on an epoch boundary
D_IN, HIDDEN = 8, 32           # the model: D_IN inputs -> HIDDEN hidden units -> 1 output
DROPOUT = 0.1                  # dropout probability inside the model
SCALE_VALUE = 0.5              # the constant the persistent buffer `scale` holds
BASE_LR, WEIGHT_DECAY = 0.01, 0.01
WARMUP = 5                     # steps of linear learning-rate warm-up before the cosine decay
MAX_NORM = 0.5                 # gradients are clipped to this global norm
JITTER = 0.05                  # each batch's features are scaled by 1 + JITTER * (u - 0.5), u from Python's random
TOLERANCE = 1e-6               # the drift above which a resume counts as visibly different
KILL_STEPS = sorted({0, 1, C - 1, C, C + 1, E - 1, E, E + 1, L - 1, L, L + 1, N - 1})
DEMO_KILL = 13                 # the step the demos kill at: not a multiple of C, and the replay crosses an epoch
DEVICE = torch.device("cpu")   # the measured experiment always runs on the CPU (section 2 says why)
CHECKPOINT_KEYS = ("step", "epoch", "consumed", "epoch_start", "rng", "py_rng", "model", "opt", "sched")

_FAILED_CHECKS: list = []
_STATUS: dict = {}          # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}      # label -> the stub whose NotImplementedError stopped it

# The exercises, in the order the notebook meets them: (label, name shown, stubs, where).
# The stubs are qualified names, so a half-finished `MLP` is told apart from any other `__init__`.
_EXERCISES = [("exercise 1", "same_storage", ("same_storage",), "section 1"),
              ("exercise 2", "pick_device", ("pick_device",), "section 2"),
              ("exercise 3", "MLP", ("MLP.__init__", "MLP.forward"), "section 3"),
              ("exercise 4", "make_loader", ("make_loader",), "section 4"),
              ("exercise 5", "train_steps", ("train_steps",), "section 5"),
              ("exercise 6", "capture and restore", ("capture", "restore"), "section 6"),
              ("exercise 7", "compare_state_dicts and kill_and_resume",
               ("compare_state_dicts", "kill_and_resume"), "section 7")]
_SHOWN = {label: shown for label, shown, _, _ in _EXERCISES}
_STUBS = {label: stubs for label, _, stubs, _ in _EXERCISES}
_UNAVAILABLE: dict = {}     # label -> why this PyTorch cannot run it (printed, and shown on the board)
if not _HAS_ACCELERATOR_API:
    _UNAVAILABLE["exercise 2"] = "this PyTorch has no torch.accelerator.current_accelerator(check_available=...)"


def _unfinished(exc: BaseException) -> str:
    """The qualified name of the function that raised NotImplementedError: the innermost frame."""
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_qualname if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (MLP)"; several exercises -> "exercises 3, 4 and 5"; a demo -> its own name."""
    if len(labels) > 1 and all(label in _SHOWN for label in labels):
        nums = [label.split()[-1] for label in labels]
        return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]
    parts = [f"{label} ({_SHOWN[label]})" if label in _SHOWN else label for label in labels]
    return parts[0] if len(parts) == 1 else ", ".join(parts[:-1]) + " and " + parts[-1]


class SimulatedKill(Exception):
    """Raised on purpose, by a hook, at the top of the step you choose. It derives from Exception, so
    a bare `except Exception:` would swallow it: exercise 5 checks that your loop lets it through."""


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 3     # an exercise's check
_DEMO_SECONDS = 30     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---

# PyTorch's one-off start-up cost. The first optimiser built in a fresh process imports a large part of PyTorch,
# which costs a noticeable fraction of a second on a quiet machine and several seconds on a busy one. The time
# limits below are set from the warm cost, so that cost is paid first, under a limit of its own: by the warm-up
# cell, or by the first check when that cell was not run (a restarted kernel, a single cell run on its own).
_WARMED = [False]


def _warm_up() -> None:
    """One throwaway optimiser step, state_dict round trip and torch.save / torch.load round trip."""
    layer = nn.Linear(2, 1)
    optimiser = torch.optim.AdamW(layer.parameters(), lr=0.1)
    optimiser.zero_grad()
    layer(torch.ones(1, 2)).sum().backward()
    optimiser.step()
    saved = optimiser.state_dict()
    optimiser.load_state_dict(saved)
    buffer = io.BytesIO()
    torch.save({"layer": layer.state_dict(), "optimiser": saved}, buffer)
    buffer.seek(0)
    torch.load(buffer, weights_only=True)


def _ensure_warm() -> None:
    """Pay the start-up cost once per process, before any check's timer is armed; later calls return at once.
    It never stops the notebook: a warm-up that fails or runs past its limit says so in one line, and the checks
    carry on without it (fourth review)."""
    if not _WARMED[0]:
        _WARMED[0] = True
        try:
            with _time_limit(30, "the warm-up"):
                _warm_up()
        except Exception as exc:
            print(f"the warm-up did not finish ({type(exc).__name__}: {exc}); the checks carry on without it")


def _quick(seconds: float, what: str):
    """Inside a check, fail the block with an AssertionError naming `what` if it runs longer than `seconds`.

    The check as a whole has the longer limit of `_limited`; this one stops a single call of your code fast,
    because a loop that keeps appending to a list can fill gigabytes of memory in a few seconds. It nests inside
    the check's own limit. It hands back the guard itself, so `with _quick(...)` is the guard's own `with`: the
    guard then knows where its block ends, even when a bare `except:` in your loop swallows the first stop. What a
    stopped call had built is freed when the next limit starts (the guard collects it then)."""
    return _time_limit(seconds, what)


# What a check may change for the length of a block and must leave as it found it. When a loop swallows the time
# limit's first stop, the guard raises again at the next line that runs, and that line may be inside the clean-up
# of a block that was still open (PyTorch's own `no_grad`, or `_patched` below), which then never finishes. So
# `_try` takes this snapshot before every check and puts it back afterwards, whatever happened (fourth review).
_PATCHABLE = ("capture", "restore", "train_steps", "make_loader", "MAX_NORM")


def _world_now() -> tuple:
    """PyTorch's gradient mode, the module-level names the checks swap, torch.save / torch.load, and the probe on
    `Table` (the names that do not exist yet are left out)."""
    table = globals().get("Table")
    return (torch.is_grad_enabled(), {name: globals()[name] for name in _PATCHABLE if name in globals()},
            (torch.save, torch.load), (table, (table.limit, table.message, table.seen) if table else None))


def _restore_world(saved: tuple) -> None:
    grad, names, (save, load), (table, probe) = saved
    torch.set_grad_enabled(grad)
    globals().update(names)
    torch.save, torch.load = save, load
    if table is not None:
        table.limit, table.message, table.seen = probe


def _steps(state, n: int, hook=None) -> list:
    """`train_steps` as the checks call it: a few steps take a few milliseconds, so a call that is still running
    after a third of a second is a loop that never ends, and it is stopped there."""
    with _quick(0.3, "train_steps on a few steps"):
        return train_steps(state, n, hook)


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet says so, by name. A wrong answer prints the check's own message,
    which names the likely mistake, and the notebook carries on, so one broken exercise never hides
    the feedback on the others. A demo names the exercises it `needs`: until each has passed its
    check, the demo says which one it is waiting for and skips. Nothing is swallowed: every outcome is
    recorded for the progress board, and a script run exits non-zero there if any check came back wrong.
    A check that runs longer than its limit is stopped and counts as failed, so a loop that never ends
    cannot hang the notebook.
    """
    if label in _UNAVAILABLE:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, {_UNAVAILABLE[label]}.")
        return
    waiting = [name for name in needs if _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    _ensure_warm()                      # before the check's timer is armed, never inside it
    saved = _world_now()
    try:
        try:
            _limited(label, check)
        finally:
            _restore_world(saved)
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, stubs in _STUBS.items() if stub in stubs and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises NotImplementedError.")
        elif label in _SHOWN:
            print(f"{label}: not implemented yet, `{stub or 'a stub'}` still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
        else:
            print(f"{label}: skipped, `{stub or 'a stub'}` is not implemented yet.")
    except SimulatedKill as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, SimulatedKill({exc}) escaped: the code that starts a run must catch it, "
              "and only it.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## The protocol, and the order of the work

Everything the exercises share is a constant in the setup cell above, so no number below is
typed into the prose. The next cell prints them, with the properties the sizes were chosen
for: the experiment needs a checkpoint that lands in the middle of an epoch, one that lands exactly
on an epoch boundary, a run long enough to cross several epochs, and a kill step that is legal for
every value in the sweep.

The notebook goes in this order, and a cell that needs an exercise you have not finished says
which one and skips, so **Run all** always reaches the progress board at the foot:

1. tensors and views (`same_storage`), 2. devices (`pick_device`), 3. the model (`MLP`),
4. the data (`make_loader`), 5. the training loop (`train_steps`), 6. the checkpoint (`capture`,
`restore`), 7. the experiment (`compare_state_dicts`, `kill_and_resume`), then what each missing
piece costs.

In [ ]:
if __name__ == "__main__":
    print(f"table: R = {R} rows, batches of B = {B}, so E = {E} batches in an epoch (the last one holds "
          f"{R - (E - 1) * B} rows, so there is no short batch: {R % B == 0})")
    print(f"run: N = {N} steps, a checkpoint every C = {C} steps, L = lcm(C, E) = {L}")
    print(f"a checkpoint lands mid-epoch (E is not a multiple of C): {E % C != 0}")
    print(f"a checkpoint lands exactly on an epoch boundary, at step L = {L}: {L < N and L % E == 0}")
    print(f"the run crosses {N / E:.2f} epochs; every kill step in the sweep is legal (0 <= k < N): "
          f"{all(0 <= k < N for k in KILL_STEPS)}")
    print(f"the sweep of kill steps: {KILL_STEPS}")
    print(f"model: {D_IN} inputs -> {HIDDEN} hidden units (dropout {DROPOUT}) -> 1 output; "
          f"AdamW(lr={BASE_LR}, weight_decay={WEIGHT_DECAY}); gradients clipped to a global norm of {MAX_NORM}")

**Warm-up.** The first time PyTorch's optimiser, `state_dict` and `torch.save` run in a fresh
process, PyTorch imports a large part of itself, which costs a noticeable fraction of a second. The
time limits on the checks below are set from the warm cost, so this cell pays the cold cost first,
under a generous limit of its own. (If you restart the kernel and skip this cell, the first check
pays it for you, before its own timer starts.)

In [ ]:
if __name__ == "__main__":
    _ensure_warm()
    print("warm-up done")

## 1. Tensors: shape, strides, and the difference between a view and a copy

A tensor is a small descriptor (dtype, shape, strides, offset) laid over a block of memory
called its **storage**. Many operations hand back a new descriptor over the SAME block:
transposing, slicing, `view`. Those are *views*, and writing through one changes the others.
`clone()` and arithmetic allocate a new block. The **stride** of a dimension is how many
elements to skip in memory to move one step along it; a transpose only swaps strides.

Why this matters before any training: `state_dict()` (section 6) hands you tensors that are
views of the live parameters, `torch.from_numpy` shares memory with its array, and `Tensor.to`
returns the very same tensor when there is nothing to change. The next cell measures each of
those, and the trap in the obvious way to test for sharing.

In [ ]:
if __name__ == "__main__":
    x = torch.arange(12.0).reshape(3, 4)
    views = [("x", x), ("x.T", x.T), ("x[:, 0]", x[:, 0]), ("x[1:]", x[1:]), ("x.view(4, 3)", x.view(4, 3)),
             ("x.clone()", x.clone()), ("x * 1", x * 1)]
    print(f"x: dtype {x.dtype}, shape {tuple(x.shape)}, strides {x.stride()}")
    for name, t in views:
        before = x.clone()
        t.add_(100.0)                          # write through t, then look at x, then put x back
        shares = not torch.equal(x, before)
        x.copy_(before)
        print(f"{name:<14} shape {str(tuple(t.shape)):<8} strides {str(t.stride()):<8} "
              f"contiguous {str(t.is_contiguous()):<5} writing to it changes x: {shares}")
    print()
    print("broadcasting (3, 1) with (4,) gives shape", tuple(torch.broadcast_shapes((3, 1), (4,))),
          "and copies nothing: a stretched dimension gets stride", torch.ones(3, 1).expand(3, 4).stride()[1])
    # The obvious test for sharing, and why it is wrong
    empty_a, empty_b = torch.empty(0), torch.empty(0)
    print("\nthe address of the FIRST element (data_ptr), compared instead of the block:")
    print("  x[1:] against x (they share memory):                      equal addresses:", x[1:].data_ptr() == x.data_ptr())
    print("  two unrelated empty tensors (they share nothing):         equal addresses:", empty_a.data_ptr() == empty_b.data_ptr())
    print("  an empty slice x[3:3] against x (cut from it, so shared): equal addresses:", x[3:3].data_ptr() == x.data_ptr())
    # Sharing across libraries and across .to()
    array = np.arange(3.0)
    shared = torch.from_numpy(array)
    array[0] = 99.0
    print("\ntorch.from_numpy shares memory with the array it came from:", shared[0].item() == 99.0)
    layer = nn.Linear(2, 2)
    print("Tensor.to with nothing to change returns the same tensor:", x.to(x.dtype) is x)
    print("Tensor.to with a new dtype returns a new tensor:", x.to(torch.float64) is not x)
    print("Module.to returns the module itself, converted in place:", layer.to(torch.float64) is layer,
          "| its weight is now", layer.weight.dtype)

## Exercise 1 · `same_storage(a, b)`

Write the test the cell above shows the obvious attempt failing: do two tensors share one block
of memory? Comparing where each tensor's first element sits is wrong twice over, as the printed
lines show: a view that starts later in the block has a different first-element address, and an
empty tensor has no first element, so its address is the same whatever it was cut from.

The checks ask for exactly this, with a Python `bool` back:

* views share, wherever in the block they start (a transpose, a slice, a column, a reshape);
* a clone does not;
* a tensor whose block has address 0 shares with nothing, so two unrelated empty tensors do not; an empty
  slice of a tensor that holds memory DOES share with the tensor it was cut from; empty slices of two
  different tensors do not.

<details><summary>💡 Hint 1 — what to think about</summary>

A tensor is a descriptor over a storage, and the storage is the thing two views have in common.
Which of the two (the tensor, or the storage behind it) has an address that does not depend on
where the view starts? And what address does "no allocation at all" report?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Ask each tensor for the storage behind it and compare the addresses of those two storages, not of
the tensors. A tensor that owns no memory reports address zero for its storage, so two zeros must
not count as a match. A slice of a non-empty tensor still points at its parent's storage, even when
the slice is empty.

</details>

In [ ]:
def same_storage(a: torch.Tensor, b: torch.Tensor) -> bool:
    """True if `a` and `b` are backed by one block of memory, wherever in it each one starts.

    Rules:
        * views share (a transpose, a slice that starts later, a column, a reshape); a clone does not;
        * a tensor whose block has address 0 shares with nothing: two separate empty tensors do not
          share; an empty slice of a tensor that holds memory does share with that tensor;
        * return a Python bool.

    Example:
        >>> x = torch.arange(6.0)
        >>> same_storage(x, x.view(2, 3)), same_storage(x, x.clone())
        (True, False)

    Returns:
        True or False.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_same_storage() -> None:
    x = torch.arange(12.0).reshape(3, 4)
    y = torch.arange(5.0)
    cases = [
        ("x.view(4, 3)", x.view(4, 3), x, True, "a reshaped view shares the block of the tensor it came from"),
        ("x.T", x.T, x, True, "a transpose shares the block of the tensor it came from"),
        ("x[:2]", x[:2], x, True, "a slice shares the block of the tensor it came from"),
        ("x[:, 0]", x[:, 0], x, True, "a column shares the block of the tensor it came from"),
        ("x[1:], a view that starts later in the block", x[1:], x, True,
         "this view starts later, so its first element has another address although the block is the same: "
         "compare the memory blocks, not the first elements"),
        ("x.clone()", x.clone(), x, False, "a clone has a block of its own"),
        ("two separate empty tensors", torch.empty(0), torch.empty(0), False,
         "an empty tensor owns no memory, so its address is zero: two zeros must not count as a match"),
        ("an empty slice x[3:3] and x", x[3:3], x, True,
         "an empty slice still points into the block of the tensor it was cut from"),
        ("empty slices of two different tensors", x[3:3], y[5:5], False,
         "slices of two different tensors point into two different blocks, even when both are empty"),
    ]
    for name, left, right, expected, why in cases:
        got = same_storage(left, right)
        assert isinstance(got, bool), f"return a Python bool, got {type(got).__name__} for {name}"
        assert got is expected, f"same_storage said {got} for {name}; it should be {expected}: {why}"
    print("exercise 1 looks right: views share, clones and unrelated empty tensors do not")


if __name__ == "__main__":
    _try("exercise 1", _check_same_storage)

## 2. Devices: write the loop once, run it anywhere

The same script should run on a CPU or on whatever accelerator the machine has (CUDA, MPS and XPU
are examples), and the idiom for that is short: pick a device once, then move the model and each batch
to it with `.to(device)`. `torch.accelerator` is PyTorch's package for the current accelerator.

There is one trap, and it is worth an exercise. According to the PyTorch documentation (quoted in
`claims.yaml`), `current_accelerator()` returns the device of the accelerator available *at compilation
time*: it is the question "what was this build made for?", not "is there one here?". A build made for
CUDA, on a machine with no GPU, therefore still answers `cuda`, and `.to("cuda")` then fails. The
documented remedy is a run-time check, either `is_available()` or `current_accelerator(check_available=True)`.

The cell below shows both answers from a fake accelerator module that is honest about the two
questions, because a real GPU is not something a check can rely on being present. The checks use
the same fake. The training loop of exercise 5 does not call `.to(device)` on its batches: the measured
experiment always runs on the CPU, where the exact equality was measured, and nothing here has an
accelerator to test such a move against. The idiom is yours to apply in a loop that runs elsewhere.

In [ ]:
class FakeAccelerator:
    """A stand-in for `torch.accelerator` with the two functions exercise 2 may call.

    `compiled` is the backend this pretend build was compiled for (or None), `available` says whether
    a device is really there. The semantics follow the documentation quoted in `claims.yaml`:
    `current_accelerator()` reports the compile-time device unless it is asked to check at run time."""

    def __init__(self, compiled, available):
        self.compiled, self.available = compiled, available

    def current_accelerator(self, check_available: bool = False):
        if self.compiled is None or (check_available and not self.available):
            return None
        return torch.device(self.compiled)

    def is_available(self) -> bool:
        return self.compiled is not None and self.available


if __name__ == "__main__":
    build = FakeAccelerator("cuda", available=False)
    print("a CUDA build on a machine with no GPU:")
    print("  current_accelerator():                      ", build.current_accelerator())
    print("  current_accelerator(check_available=True):  ", build.current_accelerator(check_available=True))
    print("  is_available():                             ", build.is_available())
    print("\nthe experiment of this lesson always runs on", DEVICE, "whatever the machine has. The answer of the "
          "real probe depends on the machine, so it is not printed here.")

## Exercise 2 · `pick_device(accel=None)`

Return the device the loop can really use: the accelerator when one is available *now*, otherwise
the CPU. `accel` is any object with the interface of `torch.accelerator`; `None` means the real
module. Return a `torch.device`, never a string.

The checks pass a fake for every backend, available and not, and one with no accelerator at all.
The case that catches the trap is the one where the build was compiled for an accelerator and none
is there.

<details><summary>💡 Hint 1 — what to think about</summary>

Two questions hide behind "which device": which kind of accelerator was this PyTorch built for, and
is one there right now? Which of the two does `current_accelerator()` answer when you call it with
no arguments?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Ask for the run-time answer: either ask `is_available()` first and only then for the device, or pass
the keyword that makes `current_accelerator` check at run time. When that gives you nothing, return
the CPU device. Remember the default for `accel`.

</details>

In [ ]:
def pick_device(accel=None) -> torch.device:
    """The device this machine can run on right now: the accelerator if one is available, else the CPU.

    Rules:
        * `accel` has the interface of `torch.accelerator`; None means `torch.accelerator` itself;
        * a build compiled for an accelerator on a machine without one must give the CPU;
        * return a `torch.device`.

    Example:
        >>> pick_device(FakeAccelerator("mps", available=True)).type
        'mps'
        >>> pick_device(FakeAccelerator("cuda", available=False)).type
        'cpu'

    Returns:
        A torch.device.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pick_device() -> None:
    for compiled in (None, "cuda", "mps", "xpu"):
        for available in (False, True):
            got = pick_device(FakeAccelerator(compiled, available))
            assert isinstance(got, torch.device), f"return a torch.device, got {type(got).__name__}"
            expected = compiled if (compiled is not None and available) else "cpu"
            assert got.type == expected, (
                f"with a build compiled for {compiled} and a device present: {available}, the answer should be "
                f"{expected!r}, not {got.type!r}. "
                + ("current_accelerator() with no arguments reports what the build was compiled for; ask "
                   "at run time (is_available(), or check_available=True), or .to() fails on a machine "
                   "without the device." if got.type != "cpu" else
                   "an available accelerator was ignored."))
    assert isinstance(pick_device(), torch.device), "pick_device() with no argument must use the real torch.accelerator"
    print("exercise 2 looks right: a compiled-in but missing accelerator gives the CPU")


if __name__ == "__main__":
    _try("exercise 2", _check_pick_device)

## 3. `nn.Module`: parameters, buffers, and what `state_dict()` holds

A module owns **parameters** (tensors the optimiser updates), **buffers** (tensors it needs but the
optimiser leaves alone, such as BatchNorm's running mean) and sub-modules. `state_dict()` is an
ordered dict of the parameters and the *persistent* buffers, keyed by name. A buffer registered
with `persistent=False` still belongs to the module, so `.to()` moves it, but it is not in the
`state_dict()`: the documentation (quoted in `claims.yaml`) says that is the only difference.
`train()` and `eval()` flip a flag on every sub-module, and dropout is active in one and the
identity in the other.

`Module.to` converts the module **in place** and returns it. `Tensor.to` returns a **new** tensor,
or the same one when nothing changes (the cell in section 1 printed both). Ignoring the return value
is harmless for a module and a bug for a tensor. The next cell shows the rest on a toy module that is
not the one you will write.

In [ ]:
class Gain(nn.Module):
    """A toy: y = dropout(gain * x + offset), with one constant persistent and one constant non-persistent buffer."""

    def __init__(self, width: int):
        super().__init__()
        self.gain = nn.Parameter(torch.ones(width))
        self.register_buffer("offset", torch.zeros(width))
        self.register_buffer("cache", torch.zeros(width), persistent=False)
        self.drop = nn.Dropout(0.5)

    def forward(self, x):
        return self.drop(self.gain * x + self.offset)


if __name__ == "__main__":
    toy = Gain(width=100)
    print("parameters:", [name for name, _ in toy.named_parameters()])
    print("all buffers:", [name for name, _ in toy.named_buffers()])
    print("state_dict keys:", list(toy.state_dict()))
    print("the non-persistent buffer is in the module but not in the state_dict:",
          "cache" in dict(toy.named_buffers()) and "cache" not in toy.state_dict())
    torch.manual_seed(0)
    toy.train()
    flag_train, zeros_train = toy.training, int((toy(torch.ones(100)) == 0).sum())
    toy.eval()
    flag_eval, zeros_eval = toy.training, int((toy(torch.ones(100)) == 0).sum())
    print(f"after train(): training flag {flag_train}, dropout zeroed some outputs: {zeros_train > 0} | "
          f"after eval(): training flag {flag_eval}, it zeroed some outputs: {zeros_eval > 0}")
    toy.to(torch.float64)
    print("after toy.to(torch.float64) both buffers follow the module:",
          toy.offset.dtype == toy.cache.dtype == torch.float64)

## Exercise 3 · `MLP`

The model of the experiment: `D_IN` inputs go into a linear layer `fc1` with `HIDDEN` outputs, then
a ReLU, then dropout `drop` with probability `DROPOUT`, then a linear layer `fc2` with one output.
Before `fc1` the input is standardised by two constant buffers:

* `scale`, a **persistent** buffer of shape `(D_IN,)` holding `SCALE_VALUE` in every entry;
* `shift`, a **non-persistent** buffer of shape `(D_IN,)` of zeros.

So `forward(x)` computes `fc2(drop(relu(fc1((x - shift) * scale))))` for `x` of shape
`(batch, D_IN)` and returns shape `(batch, 1)`. Both buffers are constant
after construction, because nothing in this lesson changes them; that is why a resumed run's equality
never depends on restoring them, and also why `scale` still belongs in the `state_dict()` and
`shift` must not.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of the tensors in the formula does the optimiser update, which does it only read, and which of
the ones it only reads should travel with a saved model? Where does PyTorch keep the answer to the
last question?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Call the parent constructor first. Build the two linear layers and the dropout as attributes with the
names above. Register each constant tensor with `register_buffer`, and give the one that must stay out
of the saved dictionary the keyword that says so. In `forward`, read the buffers when the method runs.

</details>

In [ ]:
class MLP(nn.Module):
    """The experiment's model: D_IN inputs -> fc1 (HIDDEN) -> ReLU -> drop (DROPOUT) -> fc2 (1), after standardising.

    Attributes (the checks look for these names):
        fc1, drop, fc2   the two linear layers and the dropout;
        scale            persistent buffer, shape (D_IN,), every entry SCALE_VALUE;
        shift            non-persistent buffer, shape (D_IN,), zeros.

    Example:
        >>> MLP()(torch.zeros(4, D_IN)).shape
        torch.Size([4, 1])
    """

    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Standardise, then fc1, ReLU, dropout and fc2.

        Returns:
            A tensor of shape (batch, 1) for `x` of shape (batch, D_IN).
        """
        # YOUR CODE HERE
        raise NotImplementedError


def _hidden_zero_fraction(model: nn.Module) -> float:
    """Fraction of the inputs of `fc2` that are exactly zero when every hidden pre-activation is positive,
    so that only dropout can have zeroed them. Changes the weights of `model`: pass a throwaway."""
    seen = {}
    handle = model.fc2.register_forward_pre_hook(lambda module, inputs: seen.update(h=inputs[0].detach().clone()))
    try:
        torch.manual_seed(0)
        with torch.no_grad():
            model.fc1.weight.fill_(1.0)
            model.fc1.bias.fill_(1.0)
            model.train()
            model(torch.ones(256, D_IN))
    finally:
        handle.remove()
    return float((seen["h"] == 0).float().mean())


def _check_mlp() -> None:
    model = MLP()
    assert isinstance(model, nn.Module), "MLP must subclass nn.Module"
    expected = D_IN * HIDDEN + HIDDEN + HIDDEN * 1 + 1
    got = sum(p.numel() for p in model.parameters())
    assert got == expected, (f"{got} parameters, expected {expected} (a D_IN x HIDDEN layer, its bias, a "
                             "HIDDEN x 1 layer and its bias): are the buffers registered as buffers, not Parameters?")
    keys = list(model.state_dict())
    assert "shift" not in keys, ("`shift` is in the state_dict(): register it with persistent=False, so it is "
                                 "part of the module but not of what gets saved")
    assert "scale" in keys, ("`scale` is missing from the state_dict(): a plain attribute is not saved; "
                             "register_buffer makes it a persistent buffer")
    assert sorted(keys) == sorted(["scale", "fc1.weight", "fc1.bias", "fc2.weight", "fc2.bias"]), (
        f"state_dict() holds {sorted(keys)}; expected scale, fc1.weight, fc1.bias, fc2.weight and fc2.bias")
    buffers = dict(model.named_buffers())
    assert set(buffers) == {"scale", "shift"}, (f"the module's buffers are {sorted(buffers)}; both `scale` and "
                                                "`shift` must be registered buffers")
    assert torch.equal(model.scale, torch.full((D_IN,), SCALE_VALUE)), "`scale` must hold SCALE_VALUE in every entry"
    assert torch.equal(model.shift, torch.zeros(D_IN)), "`shift` must be zeros of shape (D_IN,)"
    x = torch.randn(64, D_IN, generator=torch.Generator().manual_seed(0))
    model.eval()
    assert not model.training and not model.drop.training, "eval() must switch dropout off"
    first, second = model(x), model(x)
    assert first.shape == (64, 1), f"the output has shape {tuple(first.shape)}; it must be (batch, 1)"
    assert torch.equal(first, second), "in eval mode two forward passes of the same input must agree"
    for scale_value, shift_value in ((SCALE_VALUE, 0.0), (2.0, 1.0)):
        model.scale.fill_(scale_value)
        model.shift.fill_(shift_value)
        hidden = torch.relu(((x - model.shift) * model.scale) @ model.fc1.weight.T + model.fc1.bias)
        by_hand = hidden @ model.fc2.weight.T + model.fc2.bias
        assert torch.allclose(model(x), by_hand, rtol=0, atol=1e-5), (
            "forward does not equal fc2(relu(fc1((x - shift) * scale))): is it reading the buffers when it runs, "
            "and in this order?")
    model.train()
    assert model.training and model.drop.training, "train() must switch dropout on"
    assert not torch.equal(model(x), model(x)), "in train mode dropout must make two forward passes differ"
    fraction = _hidden_zero_fraction(MLP())
    assert 0.75 * DROPOUT < fraction < 1.25 * DROPOUT, (f"dropout zeroed {fraction:.3f} of the hidden units; it should be "
                                                      f"about DROPOUT = {DROPOUT}")
    model.to(torch.float64)
    assert model.scale.dtype == model.shift.dtype == torch.float64, (
        "Module.to must carry both buffers along: register_buffer both, rather than keeping a plain tensor attribute")
    MLP().load_state_dict(MLP().state_dict(), strict=True)
    print(f"exercise 3 looks right: {expected} parameters, state_dict keys {keys}")


if __name__ == "__main__":
    _try("exercise 3", _check_mlp)

## 4. Data: a pure `Dataset`, and a shuffle you own

A `Dataset` answers "give me row `i`"; a `DataLoader` turns it into shuffled batches. The `Table`
below is a *pure* map-style dataset: its rows come once from numpy's random generator (never from
PyTorch's, because the documentation promises no reproducible results across releases), `torch.from_numpy`
shares that memory, and `__getitem__` draws no random numbers. So the order of the data is decided by
the loader alone.

The loader's randomness has to be one you own. The documentation (quoted in `claims.yaml`) says the
`generator` argument is the RNG that `RandomSampler` uses to generate random indexes. The cell after
exercise 4 measures what a loader without one does to PyTorch's global random state, which dropout also
draws from. And `num_workers=0`: a worker is a separate process, and on macOS and Windows PyTorch
starts it with `spawn()` (also quoted). This lesson keeps data loading in the main process, and the
checks reject a loader with workers before they iterate it.

`Table` also counts the rows it hands out (`Table.reads`), so a loop that never stops fails a
check with a message instead of hanging the notebook.

In [ ]:
class Table(Dataset):
    """R rows of D_IN float32 features and one float32 target, generated from numpy's random generator.

    The class attributes are the checks' probe: `reads` counts every row fetched, `limit` (when set) makes
    the fetch after that many reads raise AssertionError(message), and `seen` (when a list) collects the
    index of every fetched row."""

    reads = 0
    limit = None
    message = ""
    seen = None

    def __init__(self, seed: int):
        rng = np.random.default_rng(seed)
        x = rng.standard_normal((R, D_IN)).astype(np.float32)
        w = rng.standard_normal(D_IN)
        y = (np.sin(x @ w) + 0.1 * rng.standard_normal(R)).astype(np.float32)
        self.x, self.y = torch.from_numpy(x), torch.from_numpy(y)

    def __len__(self) -> int:
        return R

    def __getitem__(self, i):
        Table.reads += 1
        if Table.limit is not None and Table.reads > Table.limit:
            raise AssertionError(Table.message)
        if Table.seen is not None:
            Table.seen.append(int(i))
        return self.x[i], self.y[i]


@contextlib.contextmanager
def _read_budget(max_reads: int, message: str):
    """Within the block, fetching more than `max_reads` more rows raises AssertionError(message)."""
    saved = (Table.limit, Table.message)
    Table.limit, Table.message = Table.reads + max_reads, message
    try:
        yield
    finally:
        Table.limit, Table.message = saved

## Exercise 4 · `make_loader(seed, g)`

Build the experiment's loader: `Table(seed)` in shuffled batches of `B` rows, keeping a short last
batch if there is one (`drop_last=False`), loaded in the main process (`num_workers=0`), shuffled with
**your generator `g`**. It is one line of PyTorch and every part of it matters later: the checkpoint
of exercise 6 is only able to bring back the data order because the loader's randomness lives in `g`.

<details><summary>💡 Hint 1 — what to think about</summary>

Which object holds the randomness of the shuffle? If it is PyTorch's global stream, what else is
drawing from it, and what happens to the data order when a model with dropout takes a different number
of draws between two runs?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

`DataLoader` takes the dataset, the batch size, a flag that turns shuffling on, the number of worker
processes, whether to drop a short last batch, and the generator to shuffle with. Pass the generator
you were given rather than creating one.

</details>

In [ ]:
def make_loader(seed: int, g: torch.Generator) -> DataLoader:
    """The experiment's loader over `Table(seed)`, shuffled with the generator `g`.

    Rules:
        * batches of B rows, a short last batch kept (drop_last=False), shuffled;
        * num_workers=0: the data is loaded in this process;
        * the shuffle must draw from `g`, never from PyTorch's global stream.

    Example:
        >>> len(make_loader(0, torch.Generator().manual_seed(1)))
        7

    Returns:
        A DataLoader.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_make_loader() -> None:
    g = torch.Generator().manual_seed(123)
    loader = make_loader(3, g)
    # Inspect the settings before anything is iterated: a loader with workers must never be iterated here.
    assert isinstance(loader, DataLoader), f"return a DataLoader, got {type(loader).__name__}"
    assert loader.num_workers == 0 and not loader.persistent_workers, (
        f"num_workers is {loader.num_workers}: this lesson loads data in the main process, because a worker is "
        "a separate process (and under spawn it would re-run the notebook)")
    assert loader.batch_size == B, f"batch_size is {loader.batch_size}, it must be B = {B}"
    assert loader.drop_last is False, "drop_last must be False"
    assert isinstance(loader.sampler, RandomSampler), "shuffle=True is missing: the sampler is not a RandomSampler"
    assert loader.generator is g, ("the loader must be built with generator=g, so that the shuffle draws from the "
                                   "generator you were given and not from PyTorch's global stream")
    assert len(loader) == E, f"the loader has {len(loader)} batches, expected E = {E}"
    global_before = torch.get_rng_state()
    batches = list(loader)
    assert torch.equal(torch.get_rng_state(), global_before), (
        "iterating the loader moved PyTorch's global random state: the shuffle is not using the generator g")
    assert len(batches) == E and all(xb.shape == (B, D_IN) and yb.shape == (B,) for xb, yb in batches), (
        "each batch must be (B, D_IN) features and (B,) targets")
    rows = {row.numpy().tobytes() for xb, _ in batches for row in xb}
    table = Table(3)
    assert rows == {row.numpy().tobytes() for row in table.x}, "one epoch must show every row of Table(seed) exactly once"
    assert torch.equal(loader.dataset.x, table.x), "the dataset must be Table(seed) for the seed you were given"
    again = list(make_loader(3, torch.Generator().manual_seed(123)))
    assert all(torch.equal(a[0], b[0]) for a, b in zip(batches, again)), "the same generator state must give the same order"
    other = next(iter(make_loader(3, torch.Generator().manual_seed(124))))
    assert not torch.equal(other[0], batches[0][0]), "a different generator seed must give a different order"
    print("exercise 4 looks right: the shuffle draws from your generator and one epoch shows every row once")


if __name__ == "__main__":
    _try("exercise 4", _check_make_loader)

In [ ]:
# When does a loader draw from its generator? Measured on YOUR loader, a state snapshot at a time. Exercise 5's
# way of ending an epoch is built on what these lines say.
def _demo_generator_draws() -> None:
    g = torch.Generator().manual_seed(SEED + 1)
    loader = make_loader(SEED, g)
    before_iter = g.get_state()
    it = iter(loader)
    after_iter = g.get_state()
    next(it)
    after_first = g.get_state()
    for _ in range(E - 1):
        next(it)
    after_last = g.get_state()
    try:
        next(it)
        stopped = False
    except StopIteration:
        stopped = True
    after_stop = g.get_state()
    for label, value in ((f"creating the iterator draws from the generator", not torch.equal(before_iter, after_iter)),
                         ("the first batch draws from it (the epoch's shuffle)", not torch.equal(after_iter, after_first)),
                         (f"batches 2 to {E} draw nothing", torch.equal(after_first, after_last)),
                         ("one more pull raises StopIteration", stopped),
                         ("and that pull draws from the generator too", not torch.equal(after_last, after_stop))):
        print(f"{label:<66}{value}")
    plain = DataLoader(Table(SEED), batch_size=B, shuffle=True)
    torch.manual_seed(0)
    start = torch.get_rng_state()
    list(plain)
    print(f"{'a loader built WITHOUT a generator moved the global random state':<66}{not torch.equal(torch.get_rng_state(), start)}")
    torch.set_rng_state(start)
    list(make_loader(SEED, torch.Generator().manual_seed(1)))
    print(f"{'your loader, for a whole epoch, moved it':<66}{not torch.equal(torch.get_rng_state(), start)}")


if __name__ == "__main__":
    _try("when the loader draws", _demo_generator_draws, needs=("exercise 4",))

## 5. The training loop, and the state it carries

Everything a run is, in one object. Besides the model, the optimiser and the scheduler, the run
has a **data position**: which epoch it is in, where that epoch's shuffle started
(`epoch_start`, a snapshot of the loader's generator taken *before* the epoch's iterator is created),
how many batches of the epoch have been used (`consumed`), and the live iterator `it`. The cell below
is given: the constructors, `build_state`, and a helper that swaps a module-level name for the length
of a block (the experiments use it to run your code with one deliberately wrong piece).

A step has a fixed order, and the order is the lesson's subject as much as any API: **clear** the
gradients, **forward**, **loss**, **backward**, **clip** the global gradient norm, **optimiser** step,
**scheduler** step. Get any of those out of order and the run still runs, which is why the checks below
watch the *order* with probes on the optimiser rather than waiting for a number to look wrong or
comparing numbers measured on one PyTorch release.

In [ ]:
def lr_lambda(step: int) -> float:
    """Warm-up for WARMUP steps, then a cosine to zero at step N. A plain function (a lambda would do as well):
    LambdaLR saves None for a function, so the scheduler's state_dict loads with weights_only=True."""
    if step < WARMUP:
        return (step + 1) / WARMUP
    return 0.5 * (1.0 + math.cos(math.pi * (step - WARMUP) / (N - WARMUP)))


def make_model() -> nn.Module:
    """A fresh model, on DEVICE. Calls your MLP."""
    return MLP().to(DEVICE)


def make_optimiser(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)


def make_scheduler(optimiser: torch.optim.Optimizer):
    """Built straight after the optimiser: building it writes the epoch-0 learning rate into the optimiser."""
    return torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)


def make_state(model, opt, sched, g, loader, it, epoch, epoch_start, consumed, step) -> SimpleNamespace:
    """The one place the field names of a run are written down."""
    return SimpleNamespace(model=model, opt=opt, sched=sched, g=g, loader=loader, it=it, epoch=epoch,
                           epoch_start=epoch_start, consumed=consumed, step=step)


def build_state(seed: int = SEED) -> SimpleNamespace:
    """A fresh run at step 0: seeds both global generators, builds model, optimiser and scheduler, a generator
    for the loader, and the loader; records the epoch's start BEFORE creating the iterator."""
    torch.manual_seed(seed)
    random.seed(seed)
    model = make_model()
    opt = make_optimiser(model)
    sched = make_scheduler(opt)
    g = torch.Generator().manual_seed(seed + 1)
    loader = make_loader(seed, g)
    epoch_start = g.get_state()
    return make_state(model, opt, sched, g, loader, iter(loader), 0, epoch_start, 0, 0)


@contextlib.contextmanager
def _patched(**replacements):
    """For the length of the block, replace module-level names (functions or constants) by `replacements`."""
    namespace = globals()
    saved = {name: namespace[name] for name in replacements}
    namespace.update(replacements)
    try:
        yield
    finally:
        namespace.update(saved)


def _pack(ckpt: dict) -> bytes:
    """A checkpoint as the bytes torch.save would write to a file."""
    buffer = io.BytesIO()
    torch.save(ckpt, buffer)
    return buffer.getvalue()


def _unpack(blob: bytes) -> dict:
    """The checkpoint back from its bytes, with the restricted loader."""
    return torch.load(io.BytesIO(blob), weights_only=True)

**The loss-shape trap.** The model returns shape `(B, 1)` and the targets have shape `(B,)`. PyTorch
broadcasts them against each other rather than refusing, so `F.mse_loss(pred, target)` quietly averages
a `(B, B)` table of every prediction against every target. It warns, but a warning scrolls past. The
cell prints the shape it broadcasts to, the two values, and whether PyTorch warned.

In [ ]:
if __name__ == "__main__":
    pred = torch.linspace(0.0, 1.0, B).reshape(B, 1)
    target = torch.linspace(1.0, 0.0, B) ** 2
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        broadcast_loss = F.mse_loss(pred, target)
    matching_loss = F.mse_loss(pred.squeeze(-1), target)
    print("prediction", tuple(pred.shape), "against target", tuple(target.shape), "broadcast to",
          tuple(torch.broadcast_shapes(pred.shape, target.shape)))
    print("PyTorch warned about the mismatch:", bool(caught))
    print("the loss over the broadcast table equals the loss over matching shapes:",
          bool(torch.isclose(broadcast_loss, matching_loss)))

## Exercise 5 · `train_steps(state, n, hook=None)`

Run `n` more training steps on `state`, in place, and return the `n` losses as Python floats. The
docstring lists the order of a step. These details decide whether the checkpoint of exercise 6 can work,
so read them twice:

* the epoch ends at the pull that **raises `StopIteration`**. You record `state.epoch_start` before you
  create the next iterator, because creating it and the first pull both draw from the generator, and a
  restore has to replay exactly those draws. This is the lesson's rule; a loop that rolls the epoch another
  way (for example as soon as `consumed` reaches `E`) is accepted too, as long as `(epoch_start, consumed)`
  replays to the next batch it serves, because every comparison here is with your own run;
* a hook is called at the **top** of each step, before anything else in it, even before a random number is
  drawn: a checkpoint taken in the hook has to see both generators as the step is about to use them. The
  harness of exercise 7 uses it to take checkpoints and to raise `SimulatedKill`; your loop must let that
  exception through;
* the batch stays where the model is, on `DEVICE` (the CPU): do not move it with `.to(...)`. The exact
  equality was measured on the CPU, and no check here has an accelerator to move it to;
* `MAX_NORM` is a module-level constant: read it when the step runs, because the checks change it.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of the calls in a step write gradients, which read them, and which move the weights? What would
the clip do to gradients that had not been cleared first, or if it ran after the optimiser had already
used them? And what shape does `mse_loss` need on each side?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Loop `n` times. At the top call the hook if there is one. Pull a batch inside a `try` that catches only
`StopIteration`; in its handler do the four bookkeeping updates in the order the docstring gives, then
pull again. Then the numbered updates, in order, with the prediction reshaped to match the target.
`loss.item()` is the float to keep. Do not wrap the step in any other `try`.

</details>

In [ ]:
def train_steps(state: SimpleNamespace, n: int, hook=None) -> list:
    """Run `n` more training steps on `state` in place; return the n losses as Python floats.

    A step, in this order:
        1. `hook(state)` if a hook was given, before anything else in the step;
        2. pull a batch with `next(state.it)`. When that raises StopIteration the epoch is over: do
           `state.epoch += 1`, `state.epoch_start = state.g.get_state()`, `state.it = iter(state.loader)`,
           `state.consumed = 0` (in that order), and pull again. After every pull `state.consumed += 1`;
        3. draw `u = random.random()` once and scale the batch features by `1 + JITTER * (u - 0.5)`;
        4. `state.opt.zero_grad()`, forward pass, mean squared error between the prediction and the target
           (the model's output has shape (B, 1), the target (B,)), `backward()`, clip the global gradient
           norm of the model's parameters to MAX_NORM, `state.opt.step()`, `state.sched.step()`;
        5. `state.step += 1` and keep `loss.item()`.
    Do not catch anything but StopIteration: a SimulatedKill raised by the hook must reach the caller.

    Example:
        >>> state = build_state()
        >>> losses = train_steps(state, 3)
        >>> len(losses), state.step, state.consumed
        (3, 3, 3)

    Returns:
        The list of n float losses.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _grad_norm(model: nn.Module) -> float:
    """The global L2 norm of every gradient the model holds, as a float."""
    norms = [p.grad.detach().norm() for p in model.parameters() if p.grad is not None]
    return float(torch.stack(norms).norm()) if norms else 0.0


class _StepProbe:
    """Watches one run from the outside. At every `optimiser.step()` it records the learning rate in use and
    the global gradient norm; it counts backward passes that found gradients left over from an earlier one;
    it records the norm each backward pass produced before anything clipped it; and its `hook` method records
    the step a hook saw and how many optimiser steps had happened by then. A context manager: it puts the
    optimiser back as it found it."""

    def __init__(self, state):
        self.state = state
        self.lrs, self.norms, self.raw_norms = [], [], []
        self.hook_steps, self.opt_calls_at_hook = [], []
        self.stale = 0
        self._pending = 0
        self._handles = []

    def __enter__(self):
        opt, probe = self.state.opt, self
        real_step = opt.step

        @functools.wraps(real_step)
        def step(*args, **kwargs):
            probe.lrs.append(opt.param_groups[0]["lr"])
            probe.norms.append(_grad_norm(probe.state.model))
            return real_step(*args, **kwargs)

        self._real_step, opt.step = real_step, step
        params = list(self.state.model.parameters())
        for p in params:
            self._handles.append(p.register_hook(functools.partial(self._on_grad, p)))
            self._handles.append(p.register_post_accumulate_grad_hook(functools.partial(self._on_accumulated, len(params))))
        return self

    def _on_grad(self, p, grad):
        if p.grad is not None and bool((p.grad != 0).any()):
            self.stale += 1

    def _on_accumulated(self, how_many, p):
        self._pending += 1
        if self._pending == how_many:
            self._pending = 0
            self.raw_norms.append(_grad_norm(self.state.model))

    def hook(self, state):
        self.hook_steps.append(state.step)
        self.opt_calls_at_hook.append(len(self.lrs))

    def __exit__(self, *exc):
        for handle in self._handles:
            handle.remove()
        self.state.opt.step = self._real_step
        return False


def _check_train_steps() -> None:
    n = 6
    state = build_state()
    before = (torch.get_rng_state(), random.getstate())
    at_hook = []                          # the model and both generators as the hook saw them at the top of step 0
    Table.seen = []

    def watching(s):
        probe.hook(s)
        if s.step == 0:
            at_hook.append((copy.deepcopy(s.model), torch.get_rng_state(), random.getstate()))
    try:
        with _StepProbe(state) as probe, _read_budget((n + 1) * B, "train_steps fetched more rows than n steps need: "
                                                      "does a loop pull batches without stopping?"):
            losses = _steps(state, n, watching)
        first_rows = list(Table.seen[:B])
    finally:
        Table.seen = None
    assert isinstance(losses, list) and len(losses) == n, f"return the list of the n = {n} losses"
    assert all(type(v) is float for v in losses), (
        "return loss.item() (a Python float) for every step: a loss tensor keeps its whole graph alive")
    assert state.step == n, f"state.step is {state.step} after {n} steps"
    assert probe.hook_steps == list(range(n)) and probe.opt_calls_at_hook == list(range(n)), (
        f"the hook must be called once at the top of every step, before the optimiser has moved in that step; "
        f"it saw steps {probe.hook_steps}")
    assert len(probe.lrs) == n, f"optimiser.step() ran {len(probe.lrs)} times in {n} steps; it must run once per step"
    assert state.sched.last_epoch == n, f"scheduler.step() ran {state.sched.last_epoch} times in {n} steps; once per step"
    for i, lr in enumerate(probe.lrs):
        assert abs(lr - BASE_LR * lr_lambda(i)) < 1e-12, (
            f"at step {i} the optimiser stepped with learning rate {lr}, not the schedule's {BASE_LR * lr_lambda(i)}: "
            "call scheduler.step() after optimiser.step(), once per step")
    assert probe.stale == 0, ("a backward pass found gradients left over from the step before: "
                              "call optimiser.zero_grad() before backward()")
    assert at_hook and torch.equal(at_hook[0][1], before[0]) and at_hook[0][2] == before[1], (
        "the step drew a random number before the hook ran, so a checkpoint taken in the hook would miss it: "
        "call the hook first, then pull a batch, then draw")
    expected = _first_loss_by_hand(state.loader.dataset, first_rows, *at_hook[0])
    assert abs(losses[0] - expected) <= 1e-5 * max(1.0, abs(expected)), (
        f"the first loss is {losses[0]:.6f} but recomputing it from the rows the loader served is {expected:.6f}: "
        "check the loss shapes (a (B, 1) prediction against a (B,) target broadcasts to (B, B)), that the batch is scaled "
        "by the jitter, and that the model is in training mode")
    # clipping must act, and must read MAX_NORM when the step runs
    for limit, tiny in ((1e-3, True), (1e9, False)):
        probed = build_state()
        with _patched(MAX_NORM=limit), _StepProbe(probed) as watch:
            _steps(probed, 3)
        if tiny:
            assert all(v <= limit + 1e-6 for v in watch.norms), (
                f"with MAX_NORM = {limit} the gradient norm at optimiser.step() was {max(watch.norms):.3g}: clip the "
                "gradients to MAX_NORM after backward() and before optimiser.step()")
        else:
            assert max(watch.norms) > 1e-2, ("with a huge MAX_NORM the clip must not act, but the gradients were tiny: "
                                             "read the module constant MAX_NORM instead of writing a number")
    # the data position must be replayable from (epoch_start, consumed) at any mid-epoch step
    for steps in (3, E + 3, 2 * E + 2):
        probed = build_state()
        _steps(probed, steps)
        assert 0 < probed.consumed < E, "internal: the position checks are meant to run mid-epoch"
        want, where = _next_rows_by_replay(probed), (probed.epoch, probed.consumed)
        Table.seen = []
        try:
            _steps(probed, 1)
            got = list(Table.seen)
        finally:
            Table.seen = None
        assert got == want, (
            f"after {steps} steps (epoch {where[0]}, {where[1]} batches used) the next batch is not the one that "
            "(epoch_start, consumed) leads to: record epoch_start = g.get_state() BEFORE iter(loader), and count batches")
    # a kill from the hook must reach the caller, untouched
    killed = build_state()

    def hook(s):
        if s.step == 2:
            raise SimulatedKill(2)
    try:
        _steps(killed, 5, hook)
    except SimulatedKill:
        pass
    else:
        raise AssertionError("train_steps swallowed the kill: the hook's SimulatedKill must propagate "
                             "(catch only StopIteration in the loop)")
    assert killed.step == 2, f"the kill came at the top of step 2, so exactly 2 steps are done; state.step is {killed.step}"
    assert _steps(build_state(), 0) == [], "n = 0 runs no steps"
    print("exercise 5 looks right: the order of a step, the clip, the schedule and the data position all check out")


def _first_loss_by_hand(table: Table, rows: list, model: nn.Module, torch_state, python_state) -> float:
    """What the first step's loss must be, recomputed from outside: the rows the loader really served, the same
    jitter draw, the same dropout stream, and the mean squared error over matching shapes. `model`, `torch_state`
    and `python_state` are the run's model (a copy) and both global generators as the hook saw them at the top
    of the step. Leaves the global generators as it found them."""
    now = torch.get_rng_state(), random.getstate()
    random.setstate(python_state)
    u = random.random()
    torch.set_rng_state(torch_state)
    xb, yb = table.x[rows], table.y[rows]
    pred = model(xb * (1.0 + JITTER * (u - 0.5)))
    value = float(((pred.squeeze(-1) - yb) ** 2).mean().detach())
    torch.set_rng_state(now[0])
    random.setstate(now[1])
    return value


def _next_rows_by_replay(state) -> list:
    """The row indices the next batch of `state` must hold, found by replaying its data position from scratch
    with a second generator: start at `epoch_start`, create the iterator, skip the batches already used."""
    g = torch.Generator()
    g.set_state(state.epoch_start)
    loader = make_loader(SEED, g)
    Table.seen = []
    try:
        it = iter(loader)
        for _ in range(state.consumed):
            next(it)
        Table.seen.clear()
        next(it)
        return list(Table.seen)
    finally:
        Table.seen = None


if __name__ == "__main__":
    _try("exercise 5", _check_train_steps)

In [ ]:
# Your loop on the whole run, seen from outside by the same probe the check used: where the clip bites, and the
# schedule it ran on. Every figure is computed from your run.
def _demo_clipping_and_schedule() -> None:
    state = build_state()
    with _StepProbe(state) as probe:
        losses = train_steps(state, N)
    acted = [i for i, raw in enumerate(probe.raw_norms) if raw > MAX_NORM]
    print(f"clipping to a global norm of {MAX_NORM} acted on {len(acted)} of {N} steps")
    print(f"it acted on some steps and not on all of them: {0 < len(acted) < N}")
    print(f"the norm at optimiser.step() never exceeded MAX_NORM: {all(v <= MAX_NORM + 1e-6 for v in probe.norms)}")
    print(f"learning rate: starts at the schedule's first value: {probe.lrs[0] == BASE_LR * lr_lambda(0)}; reaches "
          f"BASE_LR after the {WARMUP}-step warm-up: {max(probe.lrs) == BASE_LR}; last step below the first: {probe.lrs[-1] < probe.lrs[0]}")
    print(f"mean loss over the last {WARMUP} steps is below that of the first {WARMUP}: "
          f"{sum(losses[-WARMUP:]) < sum(losses[:WARMUP])}")


if __name__ == "__main__":
    _try("the clip and the schedule on a whole run", _demo_clipping_and_schedule,
         needs=("exercise 3", "exercise 4", "exercise 5"))

## 6. Checkpoints: what a run is made of, and the order to put it back

`model.state_dict()` holds the weights, but a run is more than its weights. To resume *exactly* you
save: the **model**; the **optimiser** (its per-parameter state, whose entries for AdamW the next cell
prints; the documentation, quoted in `claims.yaml`, says its `state_dict` holds the optimisation state and
that "the parameter itself is NOT saved"); the
**scheduler**; **both global random generators** (dropout draws from PyTorch's, the jitter from
Python's); and the **data position**: `step`, `epoch`, `consumed` and `epoch_start`. NumPy's
generator is used once, to make the table, so it is not saved. The keys are `CHECKPOINT_KEYS`,
printed below.

Facts to measure before you write a line, in the next cell: `state_dict()` returns references,
not copies; `Module.load_state_dict` copies but `Optimizer.load_state_dict` does not, so training after
a restore edits the dict you restored from; the scheduler's constructor writes the epoch-0
learning rate into the optimiser; and `torch.load(..., weights_only=True)`, which accepts tensors,
primitive types and dictionaries (quoted), takes the whole checkpoint but not numpy's legacy
generator state. From PyTorch 2.6 `torch.load` uses `weights_only=True` when no pickle module is passed
(quoted); the lesson writes it out anyway, so that nothing depends on the default.

In [ ]:
def _loads(obj) -> bool:
    """Whether `obj` survives torch.save followed by torch.load(weights_only=True). Only the restricted
    loader's own refusal (pickle.UnpicklingError) counts as "rejected"; any other failure is a bug and shows."""
    try:
        _unpack(_pack(obj))
        return True
    except pickle.UnpicklingError:
        return False


if __name__ == "__main__":
    print("the keys of a checkpoint:", CHECKPOINT_KEYS)
    layer = nn.Linear(2, 1)
    saved = layer.state_dict()
    snapshot = {key: value.clone() for key, value in saved.items()}
    adam_probe = torch.optim.AdamW(nn.Linear(2, 1).parameters(), lr=0.1)
    adam_probe.param_groups[0]["params"][0].grad = torch.ones(1, 2)
    adam_probe.step()
    optimiser = torch.optim.SGD(layer.parameters(), lr=1.0)
    layer(torch.ones(1, 2)).sum().backward()
    optimiser.step()
    print("AdamW's state for one parameter holds:", sorted(adam_probe.state_dict()["state"][0]))
    print("\nstate_dict() entries are views of the live parameters:",
          saved["weight"].untyped_storage().data_ptr() == layer.weight.untyped_storage().data_ptr())
    print("after one optimiser step the dict taken BEFORE it has changed too:",
          not torch.equal(saved["weight"], snapshot["weight"]))
    target = nn.Linear(2, 1)
    target.load_state_dict(snapshot)
    print("Module.load_state_dict copies (no memory shared with the dict):",
          target.weight.untyped_storage().data_ptr() != snapshot["weight"].untyped_storage().data_ptr())
    adam = torch.optim.AdamW(layer.parameters(), lr=0.1)
    layer(torch.ones(1, 2)).sum().backward()
    adam.step()
    adam_dict = copy.deepcopy(adam.state_dict())
    fresh = torch.optim.AdamW(nn.Linear(2, 1).parameters(), lr=0.1)
    fresh.load_state_dict(adam_dict)
    first_state = next(iter(fresh.state.values()))
    print("Optimizer.load_state_dict keeps the dict's own tensors:",
          first_state["exp_avg"] is adam_dict["state"][0]["exp_avg"])
    # the scheduler's constructor and the optimiser's learning rate
    params = list(nn.Linear(2, 1).parameters())
    before = torch.optim.AdamW(params, lr=BASE_LR)
    schedule = torch.optim.lr_scheduler.LambdaLR(before, lr_lambda)
    for _ in range(DEMO_KILL - 1):
        before.step()
        schedule.step()
    later = torch.optim.AdamW(list(nn.Linear(2, 1).parameters()), lr=BASE_LR)
    later.load_state_dict(before.state_dict())
    loaded_lr = later.param_groups[0]["lr"]
    schedule_after = torch.optim.lr_scheduler.LambdaLR(later, lr_lambda)
    built_lr = later.param_groups[0]["lr"]
    schedule_after.load_state_dict(schedule.state_dict())
    print(f"an optimiser loaded at step {DEMO_KILL - 1} holds the schedule's learning rate for that step:",
          loaded_lr == BASE_LR * lr_lambda(DEMO_KILL - 1))
    print("building a scheduler after that load resets it to the schedule's first value:",
          built_lr == BASE_LR * lr_lambda(0))
    print("and the scheduler's own load_state_dict does not put it back:", later.param_groups[0]["lr"] == built_lr)
    print("\ntorch.load(weights_only=True) accepts:")
    print("  Python's random.getstate():                ", _loads(random.getstate()))
    print("  numpy default_rng(...).bit_generator.state:", _loads(np.random.default_rng(1).bit_generator.state))
    print("  numpy np.random.get_state() (legacy):      ", _loads(np.random.get_state()))

## Exercise 6 · `capture(state)` and `restore(ckpt, seed)`

`capture(state)` returns a checkpoint dict with exactly the keys of `CHECKPOINT_KEYS`, holding
tensors, numbers and plain containers only (so `weights_only=True` accepts it) and **copies**, so
that training can carry on without the checkpoint changing underneath you.

`restore(ckpt, seed)` returns a new state, built from **fresh objects** made with the lesson's own
constructors (`make_model`, `make_optimiser`, `make_scheduler`, `make_loader`), with everything loaded
back. Order matters in these places, and the checks and exercise 7 measure each:

* build the scheduler *before* `opt.load_state_dict`, as `build_state` does; a scheduler built later
  resets the optimiser's learning rate to the schedule's first value;
* the data position is `epoch_start` (put into a new generator), a new iterator, and `consumed` batches
  skipped; make the loader with that generator. Set the two global generators **last**, so that nothing
  the repositioning does can move them;
* `Module.load_state_dict(..., assign=True)` makes the parameters the checkpoint's own tensors; the
  documentation (quoted) warns the optimiser must then be created after the call. The plain call copies.

<details><summary>💡 Hint 1 — what to think about</summary>

List every place the next training step reads something that earlier steps wrote. For each, ask what
`state_dict()` (or `get_state()`) gives you, and whether it is a copy or a reference to something that
is still changing.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

In `capture`, gather the entries of `CHECKPOINT_KEYS` in a dict and copy the whole of it before returning (deep
copy, or clone each tensor). In `restore`, mirror `build_state`: model, optimiser, scheduler, then load the three
state dicts; a new generator set to the saved epoch start, the loader, an iterator, skip the batches
already used; then put the two global generators back. Return the state with `make_state`.

</details>

In [ ]:
def capture(state: SimpleNamespace) -> dict:
    """A checkpoint of `state`: a dict with the keys CHECKPOINT_KEYS, independent of the live run.

    Entries: step, epoch, consumed, epoch_start (the generator state tensor), rng (torch.get_rng_state()),
    py_rng (random.getstate()), model, opt and sched (their state_dicts). Everything in it is a tensor, a
    number or a plain container, and nothing in it changes when training continues.

    Example:
        >>> ckpt = capture(build_state())
        >>> sorted(ckpt) == sorted(CHECKPOINT_KEYS)
        True

    Returns:
        The checkpoint dict.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def restore(ckpt: dict, seed: int = SEED) -> SimpleNamespace:
    """A new state, in fresh objects, from the checkpoint `ckpt`.

    Build the model, optimiser, scheduler and loader with the lesson's constructors, load the model,
    optimiser and scheduler state, put the data position back (a new generator set to ckpt["epoch_start"],
    an iterator, ckpt["consumed"] batches skipped), then set torch's and Python's global generators from the
    checkpoint. Return the state with `make_state`; do not change `ckpt`.

    Example:
        >>> state = build_state(); _ = train_steps(state, 5)
        >>> restore(capture(state)).step
        5

    Returns:
        The restored state.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _trees_equal(a, b) -> bool:
    """Exact equality of two nested structures of dicts, lists, tuples, tensors and plain values."""
    if torch.is_tensor(a) or torch.is_tensor(b):
        return torch.is_tensor(a) and torch.is_tensor(b) and a.dtype == b.dtype and torch.equal(a, b)
    if isinstance(a, dict) and isinstance(b, dict):
        return list(a) == list(b) and all(_trees_equal(a[key], b[key]) for key in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_trees_equal(x, y) for x, y in zip(a, b))
    return a == b


def _check_capture_restore() -> None:
    state = build_state()
    _steps(state, DEMO_KILL)
    with _quick(0.3, "capture"):
        ckpt = capture(state)
    assert isinstance(ckpt, dict) and sorted(ckpt) == sorted(CHECKPOINT_KEYS), (
        f"the checkpoint holds {sorted(ckpt) if isinstance(ckpt, dict) else type(ckpt).__name__}; it needs exactly "
        f"{sorted(CHECKPOINT_KEYS)}")
    assert (ckpt["step"], ckpt["epoch"], ckpt["consumed"]) == (state.step, state.epoch, state.consumed), (
        "step, epoch and consumed must be the run's own counters")
    assert torch.equal(ckpt["epoch_start"], state.epoch_start), "epoch_start must be the run's recorded epoch start"
    assert torch.equal(ckpt["rng"], torch.get_rng_state()), "rng must be torch.get_rng_state() at capture time"
    assert ckpt["py_rng"] == random.getstate(), "py_rng must be random.getstate() at capture time"
    assert _trees_equal(ckpt["model"], dict(state.model.state_dict())), "model must be the model's state_dict()"
    assert _trees_equal(ckpt["opt"], state.opt.state_dict()), "opt must be the optimiser's state_dict()"
    assert _trees_equal(ckpt["sched"], state.sched.state_dict()), "sched must be the scheduler's state_dict()"
    loaded = _unpack(_pack(ckpt))            # raises if anything in it is not a tensor, number or plain container
    assert _trees_equal(loaded["model"], ckpt["model"]), "the model entries must survive a save and a load"
    snapshot = copy.deepcopy(ckpt)
    _steps(state, 3)
    assert _trees_equal(ckpt, snapshot), (
        "the checkpoint changed while training went on: state_dict() returns references to the live tensors, "
        "so copy them (clone each tensor or deep-copy the dict) when you capture")
    # a resume must continue exactly where the run would have gone
    for steps in (0, DEMO_KILL, L):
        live = build_state()
        _steps(live, steps)
        with _quick(0.3, "capture"):
            taken = capture(live)
        ckpt = _unpack(_pack(taken))
        untouched = copy.deepcopy(ckpt)
        saved_rng = (torch.get_rng_state(), random.getstate())
        torch.manual_seed(0)                   # scramble both generators: only a restore that sets them can put them right
        random.seed(0)
        with _read_budget((E + 1) * B, "restore fetched more rows than skipping the used batches needs: does it loop without stopping?"):
            with _quick(0.3, "restore"):
                resumed = restore(ckpt, SEED)
        assert _trees_equal(ckpt, untouched), "restore changed the checkpoint it was given: load from it, never into it"
        assert resumed.model is not live.model and resumed.opt is not live.opt, "restore must build fresh objects"
        assert (resumed.step, resumed.epoch, resumed.consumed) == (steps, live.epoch, live.consumed), (
            f"restored counters are step {resumed.step}, epoch {resumed.epoch}, consumed {resumed.consumed}; "
            f"the run had {steps}, {live.epoch}, {live.consumed}")
        assert torch.equal(torch.get_rng_state(), saved_rng[0]) and random.getstate() == saved_rng[1], (
            "after restore both global generators must hold the checkpoint's states: set them last")
        want_lr = BASE_LR * lr_lambda(steps)
        assert abs(resumed.opt.param_groups[0]["lr"] - want_lr) < 1e-15, (
            f"after restore the learning rate is {resumed.opt.param_groups[0]['lr']}, the schedule says {want_lr}: "
            "build the scheduler before loading the optimiser's state")
        assert resumed.sched.last_epoch == steps, "the scheduler's own step count must be restored too"
        _steps(resumed, N - steps)
        torch.set_rng_state(saved_rng[0])
        random.setstate(saved_rng[1])
        _steps(live, N - steps)
        differing = [key for key in live.model.state_dict()
                     if not torch.equal(live.model.state_dict()[key], resumed.model.state_dict()[key])]
        assert not differing, (
            f"resuming at step {steps} gave different weights from carrying on ({len(differing)} entries differ): "
            "something the next step reads was not saved or not put back (optimiser moments, the scheduler, "
            "a random generator, or the data position)")
    print("exercise 6 looks right: a resumed run ends where the run it was copied from ends")


if __name__ == "__main__":
    _try("exercise 6", _check_capture_restore)

## 7. The experiment: kill a run, bring it back, compare with the run that never died

A resumed run is only worth anything if it is the same run. "Close" is not good enough: a loss curve
that looks fine hides a resume that lost the optimiser's momentum or replays a different data order.
So the test is **exact**: kill the run at step `k`, resume it from the last checkpoint in brand-new
objects, finish the run, and compare every entry of the model's `state_dict()` with the final weights of
the same code run once without interruption, using `torch.equal`. The comparison is always against
*your own* uninterrupted run, never against a number measured somewhere else, because the order the
epochs roll in is a choice a correct loop may make either way.

`uninterrupted(seed)` is given. The kill is `SimulatedKill`, raised by a hook at the top of step `k`:
after `k` completed steps, after any checkpoint due at step `k`, and before anything else in the step.
It derives from `Exception` on purpose, so that a careless `except Exception:` in a loop would swallow
it; exercise 5 already checked yours does not.

In [ ]:
def uninterrupted(seed: int = SEED) -> dict:
    """The final weights (a state_dict of clones) of N steps of this code with no interruption."""
    state = build_state(seed)
    train_steps(state, N)
    return {key: value.clone() for key, value in state.model.state_dict().items()}

## Exercise 7 · `compare_state_dicts(a, b)` and `kill_and_resume(k, seed=SEED)`

**`compare_state_dicts(a, b)`** compares two state dicts with the same keys, exactly, and returns a dict:
`equal` (True when every entry passes `torch.equal`), `n_diff` (how many entries differ), `of` (how many
entries `b` holds) and `max_abs_diff` (the largest absolute difference over all entries, computed in
float64, as a Python float). A tolerance would hide the drifts this lesson is about.

**`kill_and_resume(k, seed=SEED)`** runs the experiment for one kill step and returns that dict:

1. anything but an integer with `0 <= k < N` (a negative number, `N` itself, a float, a string, a `bool`) raises
   `ValueError`: no step `N` has a top to raise the kill at;
2. `reference = uninterrupted(seed)`; build a fresh state with `build_state(seed)`;
3. a hook that, at the top of every step, calls `capture` when `state.step % C == 0`, **then** raises
   `SimulatedKill(k)` when `state.step == k`; run `train_steps(state, N, hook)` inside a `try` that
   catches `SimulatedKill` and nothing else, and raise `AssertionError` if the run was *not* killed;
4. `torch.save` the last checkpoint into a `io.BytesIO` (the moment a real job writes its file), delete
   every training object and call `gc.collect()`, then `torch.load` the bytes back with
   `weights_only=True` written out explicitly;
5. `restore` it, finish with `train_steps(resumed, N - resumed.step)`, and compare the resumed model's
   `state_dict()` with the reference.

<details><summary>💡 Hint 1 — what to think about</summary>

When a real job is killed, which of your objects are gone, and what is all that is left? The restore
must not be able to lean on anything the dead job held. And which exception may your `try` catch, so that
a genuine bug in the loop is not mistaken for the kill?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Define the hook inside the function so it can see `k` and a small dict that keeps the latest checkpoint.
Write the bytes of that checkpoint before you delete the objects, because the checkpoint, not the live
state, is what survives. For `compare_state_dicts`, count the entries `torch.equal` rejects, and take the
largest difference after converting both sides to double precision.

</details>

In [ ]:
def compare_state_dicts(a: dict, b: dict) -> dict:
    """Exact comparison of two state dicts with the same keys: {"equal", "n_diff", "of", "max_abs_diff"}.

    `equal` is True when every entry is identical (torch.equal); `n_diff` counts the entries that are not; `of`
    is the number of entries in `b`; `max_abs_diff` is the largest absolute difference over all entries,
    computed in float64, as a Python float (0.0 when everything is equal).

    Example:
        >>> w = {"w": torch.ones(2)}
        >>> compare_state_dicts(w, {"w": torch.ones(2)})["equal"]
        True

    Returns:
        The dict described above.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def kill_and_resume(k: int, seed: int = SEED) -> dict:
    """Kill a run at the top of step `k`, resume it from the last checkpoint, and compare it with the run that
    was never interrupted. Returns compare_state_dicts(resumed weights, reference weights).

    Steps: raise ValueError unless k is an int (not a bool) with 0 <= k < N; reference = uninterrupted(seed); a fresh state; a
    hook that calls capture when state.step % C == 0 and THEN raises SimulatedKill(k) when state.step == k; run
    train_steps(state, N, hook), catching SimulatedKill only (AssertionError if no kill came); torch.save the last
    checkpoint to a BytesIO; delete the training objects and gc.collect(); torch.load the bytes with
    weights_only=True; restore; train_steps(resumed, N - resumed.step); compare with the reference.

    Example:
        >>> kill_and_resume(13)["equal"]
        True

    Returns:
        The comparison dict.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _call_kill(k, seed: int = SEED) -> dict:
    """kill_and_resume(k), with a SimulatedKill that escaped it turned into a failure that names the mistake, and
    with a limit of its own: warm, one call takes a small fraction of a second."""
    try:
        with _quick(1.5, "kill_and_resume"):
            return kill_and_resume(k, seed)
    except SimulatedKill:
        raise AssertionError("SimulatedKill escaped: catch it, and only it, in kill_and_resume") from None


@contextlib.contextmanager
def _spied(obj, name: str, record: list):
    """Replace `obj.name` by a wrapper that appends (args, kwargs) to `record` and calls the original."""
    real = getattr(obj, name)

    def wrapper(*args, **kwargs):
        record.append((args, kwargs))
        return real(*args, **kwargs)
    setattr(obj, name, wrapper)
    try:
        yield
    finally:
        setattr(obj, name, real)


def _check_compare_state_dicts() -> None:
    a = {"w": torch.arange(4.0), "b": torch.zeros(2)}
    same = compare_state_dicts({key: value.clone() for key, value in a.items()}, a)
    assert same == {"equal": True, "n_diff": 0, "of": 2, "max_abs_diff": 0.0}, f"identical dicts gave {same}"
    assert type(same["equal"]) is bool and type(same["n_diff"]) is int and type(same["max_abs_diff"]) is float, (
        "equal is a bool, n_diff and of are ints, max_abs_diff is a Python float")
    last_bit = {"w": a["w"].clone(), "b": torch.nextafter(a["b"], torch.ones(2))}
    close = compare_state_dicts(last_bit, a)
    assert close["equal"] is False and close["n_diff"] == 1 and 0.0 < close["max_abs_diff"] < TOLERANCE, (
        f"one entry differs in its last bit, and the comparison must be exact: got {close}. "
        "A tolerance (allclose) would call this equal.")
    far = compare_state_dicts({"w": a["w"] + 0.5, "b": a["b"].clone()}, a)
    assert far["n_diff"] == 1 and far["max_abs_diff"] == 0.5, f"a difference of 0.5 in one entry gave {far}"
    one = compare_state_dicts({"w": torch.ones(8) + 0.5 * torch.eye(8)[3]}, {"w": torch.ones(8)})
    assert one["max_abs_diff"] == 0.5, (f"one element of eight differs by 0.5, so the largest difference is 0.5; got "
                                        f"{one['max_abs_diff']}: take the maximum over all elements, not their mean")
    wide = compare_state_dicts({"w": torch.tensor([1.0e8])}, {"w": torch.tensor([0.1])})
    want = float(torch.tensor([1.0e8]).double() - torch.tensor([0.1]).double())
    assert wide["max_abs_diff"] == want, (
        f"1e8 against 0.1 (both float32): the difference in float64 is {want!r}, got {wide['max_abs_diff']!r}. "
        "Convert both entries to float64 BEFORE subtracting: in float32 the 0.1 is lost")


def _check_kill_and_resume() -> None:
    for bad in (-1, N, N + 3, 2.0, "4", True):
        try:
            with _quick(1.5, "kill_and_resume"):
                kill_and_resume(bad)
        except ValueError:
            continue
        except NotImplementedError:
            raise                              # a stub you have not reached is not a wrong answer: _try says so
        except SimulatedKill:
            raise
        except AssertionError as exc:
            if str(exc).startswith("kill_and_resume was "):
                raise                          # stopped by its time limit: that message names the mistake
            raise AssertionError(f"kill_and_resume({bad!r}) raised AssertionError instead of ValueError: "
                                 "check k before anything else") from None
        except Exception as exc:
            raise AssertionError(f"kill_and_resume({bad!r}) raised {type(exc).__name__} instead of ValueError: "
                                 "check k before anything else") from None
        raise AssertionError(f"kill_and_resume({bad!r}) must raise ValueError: a kill step is an integer with 0 <= k < N")
    last_capture, loaded, saved, alive, killed_at = [], [], [], [], []
    real_capture, real_restore, real_steps = capture, restore, train_steps

    def watching(state, n, hook=None):
        try:
            with _quick(0.3, "train_steps"):
                return real_steps(state, n, hook)
        except SimulatedKill:
            killed_at.append((state.step, state.sched.last_epoch))
            raise

    def recording_capture(state):
        last_capture.append(state.step)
        alive.append(weakref.ref(state.model))
        with _quick(0.3, "capture"):
            return real_capture(state)

    def checking_restore(ckpt, seed=SEED):
        gc.collect()
        assert all(ref() is None for ref in alive), (
            "the training objects were still alive when restore ran: delete them and call gc.collect() first, "
            "so a restore cannot lean on anything the dead job held")
        with _quick(0.3, "restore"):
            return real_restore(ckpt, seed)

    k = C * 3                                  # a multiple of C: the checkpoint due at k is taken, then the kill comes
    with _patched(capture=recording_capture, restore=checking_restore, train_steps=watching), \
            _spied(torch, "load", loaded), _spied(torch, "save", saved), \
            _read_budget(8 * (N + E) * B, "the experiment fetched far more rows than N steps need"):
        result = _call_kill(k)
    assert isinstance(result, dict) and set(result) == {"equal", "n_diff", "of", "max_abs_diff"}, (
        f"return the dict of compare_state_dicts, got {result!r}")
    assert result["equal"] is True, f"kill_and_resume({k}) came out different from the uninterrupted run: {result}"
    assert last_capture == list(range(0, k + 1, C)), (
        f"checkpoints were taken at steps {last_capture}; expected every C = {C} steps up to and including the kill "
        f"step {k}: take the checkpoint due at step k BEFORE raising the kill")
    assert killed_at == [(k, k)], (
        f"the kill must come at the top of step {k}, after {k} completed steps and before anything else in the step; "
        f"the run was at (step, scheduler steps) = {killed_at}")
    assert saved and loaded, "the checkpoint must go through torch.save and torch.load, as a real restart would"
    assert all(kwargs.get("weights_only") is True for _, kwargs in loaded), (
        "pass weights_only=True to torch.load explicitly")
    # the harness catches the kill and nothing else
    real_train = train_steps

    def exploding(state, n, hook=None):
        if hook is not None:
            raise RuntimeError("a real bug in the loop")
        return real_train(state, n, hook)
    with _patched(train_steps=exploding):
        try:
            with _quick(1.5, "kill_and_resume"):
                kill_and_resume(DEMO_KILL)
        except (NotImplementedError, AssertionError):
            raise                              # an unstarted stub, or the call being stopped for taking too long
        except RuntimeError:
            pass
        except SimulatedKill:
            raise AssertionError("SimulatedKill escaped: catch it, and only it, in kill_and_resume") from None
        except Exception as exc:
            raise AssertionError(f"a RuntimeError from the loop came out as {type(exc).__name__}: catch SimulatedKill "
                                 "only, so that a real bug in the loop reaches you") from None
        else:
            raise AssertionError("an exception other than SimulatedKill was swallowed: catch SimulatedKill only")

    def deaf(state, n, hook=None):             # a loop that never calls the hook, so the run is never killed
        return real_train(state, n)
    with _patched(train_steps=deaf):
        try:
            with _quick(1.5, "kill_and_resume"):
                kill_and_resume(DEMO_KILL)
        except AssertionError as exc:
            if "was still running after" in str(exc):
                raise                          # the call was stopped for taking too long: that is not the answer
        except SimulatedKill:
            raise AssertionError("SimulatedKill escaped: catch it, and only it, in kill_and_resume") from None
        else:
            raise AssertionError("a run that was never killed was reported as a resume: raise AssertionError when "
                                 "train_steps returns without SimulatedKill")
    again = _call_kill(L)
    assert again["equal"] is True, f"kill_and_resume({L}) came out different from the uninterrupted run: {again}"


def _check_exercise_7() -> None:
    _check_compare_state_dicts()
    _check_kill_and_resume()
    print("exercise 7 looks right: the resumed run equals the uninterrupted one, entry for entry")


if __name__ == "__main__":
    _try("exercise 7", _check_exercise_7)

## 8. What each missing piece costs

Your `kill_and_resume` is the instrument. The next cells run it with one deliberate fault at a time,
and every row below the first is a **wrong** resume: your own `capture`, `restore` and `make_loader` with
one piece removed or one step done in the wrong order (the helper `_patched` swaps a name for the length
of one run). The first row is your right answer. Nothing here is assumed: each cell is printed by
running the experiment, and the prose that follows a table only says what the printed cells show.

First, your own correct answer at every kill step of the sweep. Every one should read `=`: the resumed run
equals the uninterrupted one, entry for entry.

In [ ]:
def _label(equal: bool) -> str:
    return "=" if equal else "x"


def _sweep_row(patches: dict) -> list:
    """kill_and_resume at every step of KILL_STEPS with `patches` applied: one comparison dict per step."""
    out = []
    for k in KILL_STEPS:
        with _patched(**patches):
            out.append(kill_and_resume(k))
    return out


def _demo_right_answer() -> None:
    row = _sweep_row({})
    print(f"{'kill step:':<14}" + " ".join(f"{k:>3}" for k in KILL_STEPS))
    print(f"{'your answer:':<14}" + " ".join(f"{_label(r['equal']):>3}" for r in row))
    print("every comparison exactly equal (largest difference 0.0):",
          all(r["equal"] and r["max_abs_diff"] == 0.0 for r in row))
    try:
        kill_and_resume(N)
    except ValueError as exc:
        print(f"kill_and_resume({N}) raises {type(exc).__name__}: there is no step {N} whose top could raise the kill")


if __name__ == "__main__":
    _try("your right answer at every kill step", _demo_right_answer,
         needs=("exercise 3", "exercise 4", "exercise 5", "exercise 6", "exercise 7"))

Now the wrong resumes. Each is one of the mistakes people make, run through your own functions:

* **forget one piece**: the optimiser, the scheduler, torch's random generator, Python's random generator,
  or the data position, each replaced by what a freshly built run holds (so a resume at step 0 is blind to
  it: a fresh run *is* a step-0 checkpoint);
* **save the generator at the wrong moment**: the loader's generator as it is now rather than as it was at
  the start of the epoch;
* **a loader that ignores its generator**, so the shuffle draws from the global stream;
* **save without copying**: `state_dict()` references, kept as they are;
* **restore twice from one dict**: the second resume starts from a dict the first resume's training moved;
* **build the scheduler after the optimiser's load**: the optimiser is reset to the schedule's first rate;
* **`assign=True` with the optimiser built first**: the optimiser holds parameters the model no longer uses;
* **roll the epoch in `restore`** when it has used `E` batches, while `train_steps` rolls at the pull that
  raises `StopIteration`: the two disagree about when an epoch ends.

The first table kills at one step, `DEMO_KILL`, whose last checkpoint is in the middle of an epoch. For each
row it prints whether the resumed run equals the uninterrupted one, how many of the entries of the model's
`state_dict()` differ, and whether the largest difference, taken in float64, exceeds `TOLERANCE`.

In [ ]:
def _variants() -> list:
    """(label, patches) for each wrong resume, built from the CURRENT capture, restore and make_loader."""
    real_capture, real_restore = capture, restore

    def without(piece):
        def restore_without(ckpt, seed=SEED):
            fresh = real_capture(build_state(seed))
            ckpt = dict(ckpt)
            if piece == "data":
                ckpt["epoch"], ckpt["consumed"], ckpt["epoch_start"] = 0, 0, fresh["epoch_start"]
            else:
                ckpt[piece] = fresh[piece]
            return real_restore(ckpt, seed)
        return {"restore": restore_without}

    def capture_midepoch(state):
        ckpt = real_capture(state)
        ckpt["epoch_start"] = state.g.get_state()
        return ckpt

    def capture_uncopied(state):
        return {"step": state.step, "epoch": state.epoch, "consumed": state.consumed,
                "epoch_start": state.epoch_start, "rng": torch.get_rng_state(), "py_rng": random.getstate(),
                "model": state.model.state_dict(), "opt": state.opt.state_dict(), "sched": state.sched.state_dict()}

    def loader_ignoring_generator(seed, g):
        return DataLoader(Table(seed), batch_size=B, shuffle=True, num_workers=0, drop_last=False)

    def restore_twice(ckpt, seed=SEED):
        first = real_restore(ckpt, seed)
        train_steps(first, N - first.step)
        return real_restore(ckpt, seed)

    def restore_late_scheduler(ckpt, seed=SEED):
        state = real_restore(ckpt, seed)
        state.sched = make_scheduler(state.opt)
        state.sched.load_state_dict(ckpt["sched"])
        return state

    def restore_assign_late(ckpt, seed=SEED):
        state = real_restore(ckpt, seed)
        state.model.load_state_dict(ckpt["model"], assign=True)
        return state

    def restore_rolls_early(ckpt, seed=SEED):
        state = real_restore(ckpt, seed)
        if state.consumed == E:
            state.epoch += 1
            state.epoch_start = state.g.get_state()
            state.it = iter(state.loader)
            state.consumed = 0
        return state

    return [("forget the optimiser state", without("opt")),
            ("forget the scheduler state", without("sched")),
            ("forget torch's random state", without("rng")),
            ("forget Python's random state", without("py_rng")),
            ("forget the data position", without("data")),
            ("save the generator mid-epoch", {"capture": capture_midepoch}),
            ("loader ignores its generator", {"make_loader": loader_ignoring_generator}),
            ("save without copying", {"capture": capture_uncopied}),
            ("restore twice from one dict", {"restore": restore_twice}),
            ("scheduler built after the load", {"restore": restore_late_scheduler}),
            ("assign=True, optimiser built first", {"restore": restore_assign_late}),
            ("restore rolls the epoch early", {"restore": restore_rolls_early})]


def _demo_wrong_resumes() -> None:
    rows = [("your right answer", {})] + _variants()
    print(f"kill at step {DEMO_KILL}; last checkpoint at step {DEMO_KILL // C * C}")
    print(f"{'resume':<38}{'equal':>7}{'entries differing':>20}{'drift above ' + format(TOLERANCE, 'g'):>20}")
    blind = []
    for label, patches in rows:
        with _patched(**patches):
            result = kill_and_resume(DEMO_KILL)
        if result["equal"] and label != "your right answer":
            blind.append(label)
        print(f"{label:<38}{str(result['equal']):>7}{str(result['n_diff']) + ' of ' + str(result['of']):>20}"
              f"{str(result['max_abs_diff'] > TOLERANCE):>20}")
    print("wrong resumes that came out EQUAL at this kill step:", blind or "none")


if __name__ == "__main__":
    _try("the wrong resumes at one kill step", _demo_wrong_resumes,
         needs=("exercise 3", "exercise 4", "exercise 5", "exercise 6", "exercise 7"))

A `=` in a wrong row is **not** a hole in the experiment. It is what a variant that is blind at that
kill step looks like, and the only way to tell blind from harmless is to sweep every kill step. The next
cells print the sweep, two wrong resumes at a time. Read each row for where it turns from `=` to `x`,
and ask what the kill steps in the `=` columns have in common: the last checkpoint of a kill step `k` is
at `c = (k // C) * C`, and the cell after the tables prints the checkpoint steps behind every `=`.

In [ ]:
_SWEEP_CACHE: dict = {}


def _sweep_group(first: int, last: int) -> None:
    variants = _variants()[first:last]
    print(f"{'kill step:':<38}" + " ".join(f"{k:>3}" for k in KILL_STEPS))
    for label, patches in variants:
        row = _SWEEP_CACHE[label] = _sweep_row(patches)
        print(f"{label:<38}" + " ".join(f"{_label(r['equal']):>3}" for r in row))


_GROUPS = ((0, 2), (2, 4), (4, 6), (6, 8), (8, 10), (10, 12))
if __name__ == "__main__":
    for _first, _last in _GROUPS:
        _try(f"the sweep, wrong resumes {_first + 1} to {_last}", lambda a=_first, b=_last: _sweep_group(a, b),
             needs=("exercise 3", "exercise 4", "exercise 5", "exercise 6", "exercise 7"))

In [ ]:
def _demo_blind_columns() -> None:
    rows = _SWEEP_CACHE
    if len(rows) < len(_variants()):
        print("skipped: the sweep tables above have not all run.")
        return
    for label, row in rows.items():
        blind = [k for k, r in zip(KILL_STEPS, row) if r["equal"]]
        steps = sorted({k // C * C for k in blind})
        if not blind:
            print(f"{label:<38} never equal: it is wrong at every kill step of the sweep")
        elif len(blind) == len(KILL_STEPS):
            print(f"{label:<38} equal at every kill step: blind in this sweep")
        else:
            print(f"{label:<38} equal at kill steps {blind}, whose last checkpoints are at steps {steps}")
    uncopied = [k for k, r in zip(KILL_STEPS, rows["save without copying"]) if r["equal"]]
    print("\nevery kill step where 'save without copying' came out equal is a multiple of C "
          f"(so the checkpoint was taken at the kill itself): {all(k % C == 0 for k in uncopied)}")
    print("and the kill steps that are multiples of C where it came out different:",
          [k for k, r in zip(KILL_STEPS, rows['save without copying']) if k % C == 0 and not r['equal']] or "none")


if __name__ == "__main__":
    _try("where each wrong resume is blind", _demo_blind_columns, needs=("the sweep, wrong resumes 11 to 12",))

**Why "restore twice from one dict" differs, measured.** `Module.load_state_dict` copies values into the
module, but `Optimizer.load_state_dict` keeps the dict's own tensors for its running averages, and the
optimiser then updates them **in place**. So a resume that trains moves the dict it was restored from. The
next cell restores from a freshly loaded dict, trains to the end, and counts how many of the optimiser's
entries in that dict moved, comparing with a second copy of the same bytes. Whether yours moves them
depends on how you wrote `restore`; the line says which.

In [ ]:
def _demo_restore_moves_the_dict() -> None:
    state = build_state()
    train_steps(state, DEMO_KILL - 1)
    blob = _pack(capture(state))
    ckpt, pristine = _unpack(blob), _unpack(blob)
    resumed = restore(ckpt)
    train_steps(resumed, N - resumed.step)
    entries = [(pid, key) for pid, entry in ckpt["opt"]["state"].items() for key in entry]
    moved = [(pid, key) for pid, key in entries if not torch.equal(ckpt["opt"]["state"][pid][key], pristine["opt"]["state"][pid][key])]
    model_moved = [key for key in ckpt["model"] if not torch.equal(ckpt["model"][key], pristine["model"][key])]
    print(f"optimiser entries of the loaded dict changed by the resumed run: {len(moved)} of {len(entries)}")
    print(f"model entries of the loaded dict changed by the resumed run:     {len(model_moved)} of {len(ckpt['model'])}")
    print("the bytes the checkpoint was saved as are unchanged:", _pack(_unpack(blob)) == blob)
    if moved:
        print("so a second resume from this loaded dict would start from moved state; resume from the bytes "
              "(or from a deep copy made inside restore) every time.")
    else:
        print("your restore copies before it loads, so a second resume from the same dict would start clean.")


if __name__ == "__main__":
    _try("what a resume does to the dict it came from", _demo_restore_moves_the_dict,
         needs=("exercise 3", "exercise 4", "exercise 5", "exercise 6"))

## 9. Self-check

1. A job saves only `model.state_dict()` every few hundred steps and is restarted after a crash. Its
   loss curve after the restart looks smooth. What can you conclude?
   - (a) The resume was exact, because the weights are the whole state of a training run, nothing more.
   - (b) Adam's running averages are stored inside the weights, so the optimiser came back too.
   - (c) Nothing about exactness yet: only an exact comparison with an uninterrupted run tells.
   - (d) PyTorch restores the optimiser automatically whenever the model's weights are loaded.

2. A script calls `torch.manual_seed(0)` once at the top. It is killed at step 300, restarted, runs the
   same seeding line, and loads the model and the optimiser. Why is the resumed run still not identical?
   - (a) Seeding puts the generators back at step 0, but by step 300 the run had advanced them.
   - (b) `torch.manual_seed` seeds only the GPU generators, so the CPU stream is never seeded at all.
   - (c) The seed is part of the model's `state_dict`, so loading it undoes the seeding.
   - (d) Dropout ignores `torch.manual_seed`; only the data shuffle follows the seed.

3. You write `model.to(device)` on one line and `batch.to(device)` on the next, and ignore what both
   return. Which statement is true?
   - (a) For a module, `.to` returns a new module and leaves the original where it was, unmoved.
   - (b) Both calls change their object in place, so ignoring the result is always safe.
   - (c) Both calls return a copy, so ignoring the result leaves both objects where they were.
   - (d) A module changes in place, but a tensor's `.to` returns a result you have to keep.

4. A colleague says `torch.load(path, weights_only=True)` "is a speed option and changes nothing about
   what can be loaded". What is true?
   - (a) It loads faster by skipping the pickle step, and accepts exactly the same files as before.
   - (b) It limits what the unpickler may build, so a file holding other objects is rejected.
   - (c) It converts every tensor it loads to float32, which makes the file smaller to hold.
   - (d) It is a safety restriction, but one that only applies to tensors saved on a GPU.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
# Questions 1, 3 and 4, measured rather than taken on trust. These lines are facts about this PyTorch build,
# printed by the code that runs here; which option is the right reading of them is for you to decide.
if __name__ == "__main__":
    toy_module, toy_tensor = nn.Linear(2, 2), torch.zeros(2)
    print(f"a checkpoint of this lesson holds {len(CHECKPOINT_KEYS)} entries: {', '.join(CHECKPOINT_KEYS)}")
    print("Module.to(...) returned the module itself:", toy_module.to(torch.float64) is toy_module,
          "| Tensor.to(...) with the dtype it already had returned the tensor itself:", toy_tensor.to(toy_tensor.dtype) is toy_tensor,
          "| with another dtype:", toy_tensor.to(torch.float64) is toy_tensor)
    print("torch.load(weights_only=True) took Python's random state:", _loads(random.getstate()),
          "| numpy's legacy state:", _loads(np.random.get_state()))

## 10. Common mistakes

* **Checkpointing a reference.** `state_dict()` returns the live tensors; copy them when you capture, or
  the "saved" checkpoint keeps moving with the run.
* **Seeding once and calling it reproducible.** What matters is the generator's state at the checkpoint,
  for PyTorch's stream, for Python's, and for the loader's own generator.
* **Recording the epoch start at the wrong moment.** `epoch_start` is the generator as it was *before*
  the epoch's iterator was created; one moment later and a replay draws different numbers.
* **A scheduler built after the optimiser's load**, which quietly resets the optimiser's learning rate.
* **A loader without its own generator**, which draws its shuffle from the global stream that dropout also uses.
* **`except Exception:` around a step**, which swallows a kill, and every other bug with it.
* **Comparing a resume with `allclose`.** A tolerance would have called several of the wrong resumes above
  equal. Compare exactly, or state the tolerance and why.
* **Reading a `=` as "this fault is harmless".** It may only mean the fault is blind at that kill step.

## 11. What this lesson does NOT cover

* **Accelerators.** The measured experiment ran on the CPU. CUDA, ROCm, XPU and MPS paths are not run
  here; `pick_device` is checked against fakes, and the answer of the real probe is not printed.
* **Reproducibility across machines and releases.** The PyTorch documentation (quoted in `claims.yaml`) says
  completely reproducible results are not guaranteed across releases, individual commits or platforms. This
  lesson MEASURED that a resumed run equals the uninterrupted one on this machine, with this PyTorch build and
  one thread. It prints booleans and counts, not weights or loss values, because those can differ in their
  last digits elsewhere. That is a measurement, not a guarantee.
* **Your PyTorch version.** Where PyTorch is already installed the launcher installs nothing, so the version
  you run may not be the one this lesson was measured with. The public checks use only invariants and
  comparisons with your own run for that reason.
* **Other platforms.** Windows and Linux on ARM are not covered by any check here.
* **Larger systems.** Mixed precision, `torch.compile`, distributed training and its checkpoints, other
  file formats such as safetensors, and data-loading workers are each their own subject.

In [ ]:
# The closing recap. Every figure is recomputed here, not quoted from the prose above.
def _recap() -> None:
    print("--- what you built, measured ---")
    print(f"  a model with {sum(p.numel() for p in make_model().parameters())} parameters and "
          f"{len(make_model().state_dict())} state_dict entries")
    row = [kill_and_resume(k) for k in (DEMO_KILL, L)]
    print(f"  killed at steps {DEMO_KILL} and {L} and resumed in fresh objects: "
          f"equal to the uninterrupted run, entry for entry: {all(r['equal'] for r in row)}")
    print("  every number above was computed by the code you just ran")


if __name__ == "__main__":
    _try("the closing recap", _recap, needs=("exercise 3", "exercise 4", "exercise 5", "exercise 6", "exercise 7"))

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, shown, stubs, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   ({_UNAVAILABLE[label]})" if label in _UNAVAILABLE else
                f"   (waiting on `{waiting}`)" if status == "not started" and waiting not in (None, "") and waiting not in stubs
                else "")
        print(f"  {_MARKS[status]}  {label} · {shown}{note}")
        if status != "passed" and next_up is None and label not in _UNAVAILABLE:
            next_up = (label, shown, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]}, {next_up[1]}, in {next_up[2]}. Its cell has hints if you are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check that ran and
    came back wrong is. In a script or under CI that ends the run non-zero, so a green exit code can never
    paper over it. In a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.